In [1]:
# Task 1: Cài đặt và nạp các thư viện cần thiết
!pip install requests beautifulsoup4 pandas

import os
import time
import hashlib
import sqlite3
import requests
import pandas as pd
from datetime import datetime
from collections import deque
from urllib.parse import urlparse, urljoin
from bs4 import BeautifulSoup

print("Import thư viện thành công!")

  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: C:\Users\kiyor\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


Import thư viện thành công!


In [2]:
# Task 1: Cấu hình tham số dành riêng cho Hacker News
TOPIC = "Technology"
SEED_URL = "https://news.ycombinator.com/"
ALLOWED_DOMAINS = ["news.ycombinator.com"]

MAX_DEPTH = 3
MAX_PAGES = 40            # Giới hạn số trang cào
REQUEST_TIMEOUT = 10      # Giây
CRAWL_DELAY = 1.0         # Giây (Crawl Politeness)
DB_PATH = "data/crawler_hackernews.db"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://www.google.com/"
}

print("=== CRAWLER CONFIGURATION ===")
print(f"Topic          : {TOPIC}")
print(f"Seed URL       : {SEED_URL}")
print(f"Allowed Domains: {ALLOWED_DOMAINS}")
print(f"Max Depth      : {MAX_DEPTH} | Max Pages: {MAX_PAGES}")
print(f"Crawl Delay    : {CRAWL_DELAY}s")

=== CRAWLER CONFIGURATION ===
Topic          : Technology
Seed URL       : https://news.ycombinator.com/
Allowed Domains: ['news.ycombinator.com']
Max Depth      : 3 | Max Pages: 40
Crawl Delay    : 1.0s


In [3]:
# Task 2, 6, 7: URL Frontier sử dụng deque (BFS) và set visited
class URLFrontier:
    def __init__(self, max_depth=3):
        self.queue = deque()
        self.visited = set()
        self.max_depth = max_depth
        self.discovered_count = 0
        self.skipped_count = 0

    def add_url(self, url, depth):
        self.discovered_count += 1
        if depth > self.max_depth or url in self.visited:
            self.skipped_count += 1
            return False
        self.visited.add(url)
        self.queue.append((url, depth))
        return True

    def get_next(self):
        return self.queue.popleft() if self.queue else None

    def is_empty(self):
        return len(self.queue) == 0

frontier = URLFrontier(max_depth=MAX_DEPTH)
print("Đã khởi tạo URL Frontier.")

Đã khởi tạo URL Frontier.


In [4]:
# Duplicate Detection: Kiểm tra exact duplicate nội dung bằng băm SHA-256
class DuplicateDetector:
    def __init__(self):
        self.fingerprints = set()

    def is_duplicate(self, text_content):
        if not text_content:
            return True
        fp = hashlib.sha256(text_content.strip().encode("utf-8")).hexdigest()
        if fp in self.fingerprints:
            return True
        self.fingerprints.add(fp)
        return False

detector = DuplicateDetector()
print("Đã khởi tạo Duplicate Detector.")

Đã khởi tạo Duplicate Detector.


In [5]:
# Task 8: Thiết lập SQLite lưu trữ bảng 'pages' và 'links'
class Database:
    def __init__(self, db_path):
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        self.conn = sqlite3.connect(db_path)
        self.create_tables()

    def create_tables(self):
        cursor = self.conn.cursor()
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS pages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                url TEXT UNIQUE,
                domain TEXT,
                title TEXT,
                content TEXT,
                depth INTEGER,
                status_code INTEGER,
                crawled_at TEXT
            )
        ''')
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS links (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                source_url TEXT,
                target_url TEXT
            )
        ''')
        self.conn.commit()

    def insert_page(self, url, domain, title, content, depth, status_code, crawled_at):
        cursor = self.conn.cursor()
        cursor.execute('''
            INSERT OR IGNORE INTO pages (url, domain, title, content, depth, status_code, crawled_at)
            VALUES (?, ?, ?, ?, ?, ?, ?)
        ''', (url, domain, title, content, depth, status_code, crawled_at))
        self.conn.commit()

    def insert_link(self, source_url, target_url):
        cursor = self.conn.cursor()
        cursor.execute('''
            INSERT INTO links (source_url, target_url)
            VALUES (?, ?)
        ''', (source_url, target_url))
        self.conn.commit()

    def close(self):
        self.conn.close()

db = Database(DB_PATH)
print("Đã kết nối SQLite Database.")

Đã kết nối SQLite Database.


In [6]:
# Task 4 & Task 5: Bóc tách text, khử nhiễu (Noise Removal) và lọc URL dành riêng cho Hacker News
IGNORED_EXTENSIONS = (
    '.jpg', '.jpeg', '.png', '.gif', '.svg', '.webp', '.ico',
    '.css', '.js', '.zip', '.tar', '.gz', '.pdf', '.mp4', '.mp3'
)

TECH_KEYWORDS = [
    'tech', 'technology', 'ai', 'artificial intelligence', 'software', 
    'hardware', 'app', 'mobile', 'apple', 'google', 'microsoft', 'nvidia',
    'cybersecurity', 'gadget', 'robotics', 'cloud', 'data', 'crypto',
    'python', 'javascript', 'linux', 'programming', 'code', 'web', 'hn', 'hacker'
]

# Các trang chức năng/tài khoản trên Hacker News cần loại bỏ
EXCLUDED_PATH_KEYWORDS = [
    'login', 'logout', 'register', 'submit', 'vote', 'flag', 
    'hide', 'fave', 'forgot', 'change', 'user', 'reply'
]

def is_valid_tech_url(url):
    """Kiểm tra URL có thuộc các trang chức năng không cần thiết hay không."""
    url_lower = url.lower()
    for excluded in EXCLUDED_PATH_KEYWORDS:
        if excluded in url_lower:
            return False
    return True

def is_tech_related(url, title, content):
    """Kiểm tra URL, tiêu đề hoặc nội dung có thuộc về chủ đề Tech không."""
    text_to_check = f"{url} {title} {content}".lower()
    matches = sum(1 for kw in TECH_KEYWORDS if kw in text_to_check)
    return matches >= 1

def extract_page_data(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    title = soup.title.get_text(strip=True) if soup.title else "Hacker News"

    # Noise Removal: loại bỏ script, style và các bảng điều hướng thừa
    for tag in soup(["script", "style", "noscript", "input", "form"]):
        tag.decompose()

    content = soup.get_text(separator=" ", strip=True)
    return title, content, soup

def extract_and_filter_links(soup, base_url, allowed_domains):
    valid_links = []
    for tag in soup.find_all("a", href=True):
        href = tag["href"].strip()
        if href.startswith(('mailto:', 'javascript:', 'tel:', '#')):
            continue

        full_url = urljoin(base_url, href)
        parsed = urlparse(full_url)

        # Lọc giao thức và domain
        if parsed.scheme not in ('http', 'https'):
            continue
        if parsed.netloc.lower() not in allowed_domains:
            continue
            
        # Lọc file tĩnh
        if any(parsed.path.lower().endswith(ext) for ext in IGNORED_EXTENSIONS):
            continue

        # Kiểm tra đường dẫn loại trừ
        if not is_valid_tech_url(full_url):
            continue

        clean_url = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
        if parsed.query:
            # Giữ lại query string vì Hacker News dùng query param ví dụ: item?id=..., p=2
            clean_url += f"?{parsed.query}"
            
        clean_url = clean_url.rstrip('/')
        if not clean_url:
            clean_url = f"{parsed.scheme}://{parsed.netloc}/"
            
        valid_links.append(clean_url)
        
    return valid_links

print("Đã thiết lập Parser hoàn chỉnh cho Hacker News.")

Đã thiết lập Parser hoàn chỉnh cho Hacker News.


In [7]:
# Task 3 & 9: Thực thi Crawler và hiển thị thống kê
session = requests.Session()
session.headers.update(HEADERS)

crawled_count = 0
failed_requests = 0
status_code_stats = {}
depth_stats = {d: 0 for d in range(MAX_DEPTH + 1)}

frontier.add_url(SEED_URL, depth=0)

print("=== BẮT ĐẦU CRAWL HACKER NEWS ===")
while not frontier.is_empty() and crawled_count < MAX_PAGES:
    current_url, current_depth = frontier.get_next()
    start_time = time.time()

    try:
        response = session.get(current_url, timeout=REQUEST_TIMEOUT)
        elapsed = time.time() - start_time
        status_code = response.status_code
    except requests.RequestException as e:
        failed_requests += 1
        print(f"[Error] {current_url}: {e}")
        continue

    status_code_stats[status_code] = status_code_stats.get(status_code, 0) + 1

    if "text/html" not in response.headers.get("Content-Type", ""):
        continue

    if status_code == 200:
        response.encoding = "utf-8"
        title, content, soup = extract_page_data(response.text)

        # 1. Kiểm tra exact duplicate
        if detector.is_duplicate(content):
            print(f"[Duplicate Content Skip] {current_url}")
            continue

        # 2. Kiểm tra ngữ nghĩa Tech
        if not is_tech_related(current_url, title, content):
            print(f"[Bỏ qua - Không liên quan Tech] {current_url}")
            continue

        # 3. Tăng biến đếm và ghi nhận dữ liệu
        crawled_count += 1
        depth_stats[current_depth] = depth_stats.get(current_depth, 0) + 1
        domain = urlparse(current_url).netloc
        crawled_at = datetime.now().isoformat()

        # 4. Lưu vào SQLite
        db.insert_page(current_url, domain, title, content, current_depth, status_code, crawled_at)

        # 5. Trích xuất và nạp link con vào Frontier
        links = extract_and_filter_links(soup, current_url, ALLOWED_DOMAINS)
        for link in links:
            db.insert_link(current_url, link)
            frontier.add_url(link, depth=current_depth + 1)

        print(f"[{crawled_count:03d}] Depth: {current_depth} | Status: {status_code} | Links: {len(links)} | Time: {elapsed:.2f}s | {current_url}")
    else:
        failed_requests += 1

    time.sleep(CRAWL_DELAY)

# In tổng kết
print("\n" + "=" * 25 + " CRAWLING SUMMARY " + "=" * 25)
print(f"Website                : {SEED_URL}")
print(f"Pages Crawled          : {crawled_count}")
print(f"Unique URLs Discovered : {frontier.discovered_count}")
print(f"Skipped URLs           : {frontier.skipped_count}")
print(f"Failed Requests        : {failed_requests}")
for d in range(MAX_DEPTH + 1):
    print(f"  - Depth {d}: {depth_stats.get(d, 0)} pages")
for code, cnt in sorted(status_code_stats.items()):
    print(f"  - HTTP {code}: {cnt}")
print("=" * 68)

=== BẮT ĐẦU CRAWL HACKER NEWS ===
[001] Depth: 0 | Status: 200 | Links: 103 | Time: 1.91s | https://news.ycombinator.com/
[Duplicate Content Skip] https://news.ycombinator.com
[002] Depth: 1 | Status: 200 | Links: 103 | Time: 0.74s | https://news.ycombinator.com/news
[003] Depth: 1 | Status: 200 | Links: 103 | Time: 0.36s | https://news.ycombinator.com/newest
[004] Depth: 1 | Status: 200 | Links: 107 | Time: 0.35s | https://news.ycombinator.com/front
[005] Depth: 1 | Status: 200 | Links: 133 | Time: 0.43s | https://news.ycombinator.com/newcomments
[006] Depth: 1 | Status: 200 | Links: 75 | Time: 0.76s | https://news.ycombinator.com/ask
[007] Depth: 1 | Status: 200 | Links: 106 | Time: 0.33s | https://news.ycombinator.com/show
[008] Depth: 1 | Status: 200 | Links: 73 | Time: 0.33s | https://news.ycombinator.com/jobs
[009] Depth: 1 | Status: 200 | Links: 103 | Time: 0.35s | https://news.ycombinator.com/from?site=openai.com
[010] Depth: 1 | Status: 200 | Links: 706 | Time: 2.46s | https:/

In [8]:
# Cell 8: Kiểm tra dữ liệu an toàn
import sqlite3
import pandas as pd

try:
    cursor = db.conn.cursor()
except (sqlite3.ProgrammingError, AttributeError):
    print("Kết nối đã bị đóng từ trước, đang kết nối lại...")
    db = Database(DB_PATH)
    cursor = db.conn.cursor()

cursor.execute("SELECT COUNT(*) FROM pages")
total_pages = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM links")
total_links = cursor.fetchone()[0]

print(f"-> Tổng số trang trong bảng 'pages': {total_pages}")
print(f"-> Tổng số liên kết trong bảng 'links': {total_links}\n")

if total_pages > 0:
    df_pages = pd.read_sql_query("SELECT id, domain, url, title, depth, status_code FROM pages LIMIT 5", db.conn)
    display(df_pages)
else:
    print("⚠️ Cảnh báo: Bảng 'pages' đang RỖNG. Hãy chạy lại Cell 7 để crawl dữ liệu!")

db.close()
print("Đã đóng kết nối cơ sở dữ liệu an toàn.")

-> Tổng số trang trong bảng 'pages': 40
-> Tổng số liên kết trong bảng 'links': 5937



,id,domain,url,title,depth,status_code
0,1,news.ycombinator.com,https://news.ycombinator.com/,Hacker News,0,200
1,2,news.ycombinator.com,https://news.ycombinator.com/news,Hacker News,1,200
2,3,news.ycombinator.com,https://news.ycombinator.com/newest,New Links | Hacker News,1,200
3,4,news.ycombinator.com,https://news.ycombinator.com/front,2026-09-22 front | Hacker News,1,200
4,5,news.ycombinator.com,https://news.ycombinator.com/newcomments,New Comments | Hacker News,1,200


Đã đóng kết nối cơ sở dữ liệu an toàn.
